In [1]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd()
if not (data_dir / "market_returns_2000_2025.csv").is_file():
    data_dir = data_dir / "CAPM Factors"

market = pd.read_csv(data_dir / "market_returns_2000_2025.csv", parse_dates=["date"])
risk_free = pd.read_csv(data_dir / "risk_free_rate_2000_2025.csv", parse_dates=["date"])

monthly = market.merge(
    risk_free,
    on="date",
    how="outer",
    validate="one_to_one",
    indicator=True,
)
unmatched = monthly.loc[monthly["_merge"] != "both", "date"]
if not unmatched.empty:
    raise ValueError(f"Market and risk-free CSV dates do not match: {unmatched.tolist()[:5]}")
if monthly[["market_return", "rf_1m"]].isna().any().any():
    raise ValueError("Market return or risk-free rate contains missing values.")

factors = pd.DataFrame(
    {
        "Mkt-RF": (monthly["market_return"] - monthly["rf_1m"]).to_numpy(),
        "RF": monthly["rf_1m"].to_numpy(),
    },
    index=pd.DatetimeIndex(monthly["date"], name="Date"),
).sort_index()



## Clean S&P 500 monthly stock returns

Use the supplied `total_return` field as the monthly total return (do not recompute it from price). Remove missing returns and exact duplicate `PERMNO`/date records, and winsorize the cross-section of returns within each month at the 1st and 99th percentiles. Keep cleaned long and wide panels in memory only. `PERMNO` identifies securities because ticker symbols can be reused.

In [2]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd()
if not (data_dir / "sp500_stocks_2000_2025.csv").is_file():
    data_dir = data_dir / "CAPM Factors"

source_path = data_dir / "sp500_stocks_2000_2025.csv"
required_columns = {"date", "permno", "ticker", "company_name", "total_return"}
source_raw = pd.read_csv(source_path, parse_dates=["date"])
stocks = source_raw.copy()
missing_columns = required_columns.difference(stocks.columns)
if missing_columns:
    raise ValueError(f"Stock data is missing required columns: {sorted(missing_columns)}")

source_rows = len(stocks)
stocks["date"] = pd.to_datetime(stocks["date"], errors="coerce")
if stocks["date"].isna().any() or stocks["permno"].isna().any():
    raise ValueError("Stock data contains missing or invalid dates or PERMNO identifiers.")
if not stocks["date"].dt.is_month_end.all():
    raise ValueError("Expected month-end observations in the stock return file.")
stocks["total_return"] = pd.to_numeric(stocks["total_return"], errors="raise")

key_columns = ["permno", "date"]
duplicate_rows = stocks.duplicated(key_columns, keep=False)
if duplicate_rows.any():
    duplicate_groups = stocks.loc[duplicate_rows].groupby(key_columns, dropna=False)
    conflicting_duplicates = duplicate_groups.nunique(dropna=False).gt(1).any(axis=1)
    if conflicting_duplicates.any():
        raise ValueError("Conflicting duplicate PERMNO/date observations found.")
duplicates_removed = int(stocks.duplicated(key_columns).sum())
stocks = stocks.drop_duplicates(key_columns).copy()

missing_returns = int(stocks["total_return"].isna().sum())
stocks = stocks.dropna(subset=["total_return"]).copy()

stocks["return_winsorized"] = stocks.groupby("date")["total_return"].transform(
    lambda monthly_returns: monthly_returns.clip(
        lower=monthly_returns.quantile(0.01),
        upper=monthly_returns.quantile(0.99),
    )
)
winsorized_returns = int(stocks["return_winsorized"].ne(stocks["total_return"]).sum())

monthly_returns = stocks[
    ["date", "permno", "ticker", "company_name", "total_return", "return_winsorized"]
].sort_values(["date", "permno"])
wide_returns = monthly_returns.pivot(
    index="date", columns="permno", values="return_winsorized"
).sort_index()
wide_returns.index.name = "date"
wide_returns.columns = [f"permno_{int(permno)}" for permno in wide_returns.columns]



## 60-month rolling beta estimates

For each stock and month-end, estimate ordinary beta using all paired stock and market excess returns in the trailing 60 calendar months, and downside beta using only paired observations where market excess return is strictly negative. Both estimates use the supplied monthly `total_return` and the aligned market and risk-free CSVs in decimal units; they do not use the winsorized return column. Missing observations are excluded, windows require at least 12 valid observations, and a missing/zero market variance yields NaN. Each estimate dated t is available after month t closes and is used to predict the next return month. The stock file provides total returns but no separate delisting-return field, so no additional delisting adjustment is applied.

In [3]:
import numpy as np
import pandas as pd

WINDOW_MONTHS = 60
MIN_OBSERVATIONS = 12

calendar = pd.DatetimeIndex(factors.index, name="date")
expected_calendar = pd.date_range(
    start=calendar.min(), end=calendar.max(), freq=pd.offsets.MonthEnd()
)
if not calendar.equals(expected_calendar):
    raise ValueError("Market data must contain one continuous observation per month-end.")
market_excess = pd.Series(factors["Mkt-RF"].to_numpy(), index=calendar)
rf_by_month = pd.Series(factors["RF"].to_numpy(), index=calendar)
full_window = pd.Series(1, index=calendar).rolling(
    WINDOW_MONTHS, min_periods=1
).count().eq(WINDOW_MONTHS)

stock_data = monthly_returns.copy()
if stock_data.duplicated(["permno", "date"]).any():
    raise ValueError("Cleaned stock data contains duplicate PERMNO/month observations.")
if not stock_data["date"].isin(calendar).all():
    raise ValueError("Stock-return dates do not all align with market months.")

results = []
for permno, stock in stock_data.groupby("permno", sort=False):
    stock_returns = stock.set_index("date")["total_return"].reindex(calendar)
    stock_excess = stock_returns - rf_by_month
    paired = stock_excess.notna() & market_excess.notna()
    downside = paired & market_excess.lt(0)

    def rolling_slope(mask):
        count = mask.astype("int64").rolling(
            WINDOW_MONTHS, min_periods=1
        ).sum()
        x = stock_excess.where(mask, 0.0)
        y = market_excess.where(mask, 0.0)
        sum_x = x.rolling(WINDOW_MONTHS, min_periods=1).sum()
        sum_y = y.rolling(WINDOW_MONTHS, min_periods=1).sum()
        sum_xy = (x * y).rolling(WINDOW_MONTHS, min_periods=1).sum()
        sum_y_squared = (y.pow(2)).rolling(
            WINDOW_MONTHS, min_periods=1
        ).sum()
        centered_cross_product = sum_xy - sum_x * sum_y / count
        centered_market_sum_squares = sum_y_squared - sum_y.pow(2) / count
        beta = centered_cross_product / centered_market_sum_squares
        beta = beta.where(
            full_window & (count >= MIN_OBSERVATIONS) & (centered_market_sum_squares > 0)
        )
        return beta, count

    beta, valid_count = rolling_slope(paired)
    downside_beta, downside_count = rolling_slope(downside)

    # Emit an estimate for each stock-month with an observed monthly stock return.
    estimation_months = pd.DatetimeIndex(stock["date"].sort_values().unique())
    theoretical_start = estimation_months - pd.offsets.MonthEnd(WINDOW_MONTHS - 1)
    window_start = pd.DatetimeIndex([max(start, calendar[0]) for start in theoretical_start])
    results.append(
        pd.DataFrame(
            {
                "permno": int(permno),
                "estimation_month": estimation_months,
                "beta": beta.reindex(estimation_months).to_numpy(),
                "downside_beta": downside_beta.reindex(estimation_months).to_numpy(),
                "n_observations": valid_count.reindex(estimation_months).to_numpy(dtype="int64"),
                "n_downside_observations": downside_count.reindex(estimation_months).to_numpy(dtype="int64"),
                "window_start": window_start,
                "window_end": estimation_months,
            }
        )
    )

rolling_betas = pd.concat(results, ignore_index=True).sort_values(
    ["estimation_month", "permno"]
)



## Downside-beta regressions: between-crises period (July 2009-January 2020)

This analysis is restricted to monthly stock-return observations from July 2009 through January 2020 (inclusive), the selected between-crises comparison period without a U.S. recession. It removes the Global Financial Crisis, COVID-19, and post-COVID return periods from the regressions and plots. This is not a claim that markets had no corrections or volatility during the interval.

For each return month, ordinary and downside betas are estimated through the immediately preceding month using trailing 60-calendar-month windows. Historical returns before 2006 may contribute to the early beta formation windows, but are not regression outcome months. This preserves full trailing-window estimates, the no-look-ahead rule, and the 12-down-month minimum for downside beta.

The two models for this period are:

\[
\text{Model 1: } R^e_{i,t}=a_t+\gamma_{1,t}\beta^-_{i,t-1}+u_{i,t}
\]

\[
\text{Model 2: } R^e_{i,t}=a_t+\gamma_{1,t}\beta^-_{i,t-1}+\gamma_{2,t}\beta_{i,t-1}+u_{i,t}
\]

The response is the supplied raw monthly stock total return less the same-month risk-free rate. The reported downside-beta estimate is the average monthly cross-sectional slope, with Newey-West/HAC standard errors using six lags, a two-sided normal-approximation p-value, and a 95% confidence interval. Model 2 estimates the downside-beta relationship controlling for ordinary beta. The output reports each model-period estimate, sample size, date range, and average cross-sectional fit statistics.

Plots show each period's stock-month observations and its fitted relationship implied by average Fama-MacBeth coefficients. Model 2 holds ordinary beta at its period sample mean. Statistical association is not causation. The supplied stock data has no separate delisting-return field and may have survivorship limitations.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HAC_LAGS = 6
TIME_PERIODS = {
    "Between crises (non-recession comparison)": (pd.Timestamp("2009-07-31"), pd.Timestamp("2020-01-31")),
}

# Pair each outcome month with beta estimates formed through the immediately prior month.
returns = monthly_returns[["permno", "date", "total_return"]].copy()
returns = returns.rename(columns={"date": "return_month", "total_return": "stock_return"})
returns["beta_formation_month"] = returns["return_month"] - pd.offsets.MonthEnd(1)

predictive = returns.merge(
    rolling_betas[["permno", "estimation_month", "beta", "downside_beta"]],
    left_on=["permno", "beta_formation_month"],
    right_on=["permno", "estimation_month"],
    how="left",
    validate="one_to_one",
)
rf = factors["RF"].rename("RF").rename_axis("return_month").reset_index()
predictive = predictive.merge(rf, on="return_month", how="left", validate="many_to_one")
if predictive["RF"].isna().any():
    raise ValueError("A return month is missing its matched risk-free rate.")
predictive["excess_return"] = predictive["stock_return"] - predictive["RF"]
predictive = predictive.replace([np.inf, -np.inf], np.nan)

model_specs = {
    "Model 1: downside beta only": ["downside_beta"],
    "Model 2: downside beta + ordinary beta": ["downside_beta", "beta"],
}
monthly_fits = {}
summary_rows = []
coefficient_rows = []

for period_name, (period_start, period_end) in TIME_PERIODS.items():
    period_data = predictive.loc[
        predictive["return_month"].between(period_start, period_end)
    ].copy()
    if period_data.empty:
        raise ValueError(f"No stock-return observations found for {period_name}.")

    for model_name, regressors in model_specs.items():
        sample = period_data.dropna(subset=["excess_return", *regressors]).copy()
        monthly_rows = []
        for month, cross_section in sample.groupby("return_month", sort=True):
            x = cross_section[regressors].to_numpy(dtype=float)
            y = cross_section["excess_return"].to_numpy(dtype=float)
            design = np.column_stack([np.ones(len(x)), x])
            if len(y) <= design.shape[1] or np.linalg.matrix_rank(design) < design.shape[1]:
                continue
            coefficients, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
            residuals = y - design @ coefficients
            total_ss = np.sum((y - y.mean()) ** 2)
            r_squared = 1 - np.sum(residuals**2) / total_ss if total_ss > 0 else np.nan
            adjusted_r_squared = (
                1 - (1 - r_squared) * (len(y) - 1) / (len(y) - design.shape[1])
            )
            row = {
                "return_month": month,
                "n_stocks": len(y),
                "alpha": coefficients[0],
                "r_squared": r_squared,
                "adjusted_r_squared": adjusted_r_squared,
            }
            row.update(dict(zip(regressors, coefficients[1:])))
            monthly_rows.append(row)

        monthly_coefficients = pd.DataFrame(monthly_rows).sort_values("return_month")
        if len(monthly_coefficients) <= HAC_LAGS + 1:
            raise ValueError(
                f"Too few valid monthly cross-sections for six-lag HAC inference "
                f"in {period_name}, {model_name}: {len(monthly_coefficients)}."
            )
        monthly_fits[(period_name, model_name)] = monthly_coefficients

        for term in ["alpha", *regressors]:
            values = monthly_coefficients[term].to_numpy(dtype=float)
            mean_estimate = values.mean()
            centered = values - mean_estimate
            n_months = len(values)
            long_run_variance = centered @ centered / n_months
            for lag in range(1, min(HAC_LAGS, n_months - 1) + 1):
                weight = 1 - lag / (HAC_LAGS + 1)
                autocovariance = centered[lag:] @ centered[:-lag] / n_months
                long_run_variance += 2 * weight * autocovariance
            standard_error = np.sqrt(max(long_run_variance, 0) / n_months)
            t_stat = mean_estimate / standard_error if standard_error > 0 else np.nan
            p_value = math.erfc(abs(t_stat) / math.sqrt(2)) if np.isfinite(t_stat) else np.nan
            coefficient_rows.append({
                "period": period_name,
                "model": model_name,
                "term": term,
                "mean_monthly_coefficient": mean_estimate,
                "hac_std_error": standard_error,
                "t_stat": t_stat,
                "p_value_normal_approx": p_value,
                "ci_95_low": mean_estimate - 1.96 * standard_error,
                "ci_95_high": mean_estimate + 1.96 * standard_error,
            })

        summary_rows.append({
            "period": period_name,
            "model": model_name,
            "requested_return_month_start": period_start,
            "requested_return_month_end": period_end,
            "observed_return_month_start": monthly_coefficients["return_month"].min(),
            "observed_return_month_end": monthly_coefficients["return_month"].max(),
            "n_stock_months": len(sample),
            "n_months": len(monthly_coefficients),
            "mean_stocks_per_month": monthly_coefficients["n_stocks"].mean(),
            "mean_r_squared": monthly_coefficients["r_squared"].mean(),
            "mean_adjusted_r_squared": monthly_coefficients["adjusted_r_squared"].mean(),
            "standard_error_method": f"Fama-MacBeth; Newey-West HAC, {HAC_LAGS} lags",
        })

regression_summary = pd.DataFrame(coefficient_rows)
model_summary = pd.DataFrame(summary_rows)

# Compare fitted downside-beta relationships across the four return windows.
for model_name, regressors in model_specs.items():
    fig, axes = plt.subplots(1, len(TIME_PERIODS), figsize=(8, 4.5), sharey=False, squeeze=False)
    for ax, (period_name, (period_start, period_end)) in zip(axes.flat, TIME_PERIODS.items()):
        period_data = predictive.loc[
            predictive["return_month"].between(period_start, period_end)
        ].dropna(subset=["excess_return", *regressors])
        coefficients = monthly_fits[(period_name, model_name)]
        average_coefficients = coefficients[["alpha", *regressors]].mean()
        x_min, x_max = period_data["downside_beta"].quantile([0.005, 0.995])
        x_line = np.linspace(x_min, x_max, 200)
        y_line = average_coefficients["alpha"] + average_coefficients["downside_beta"] * x_line
        if "beta" in regressors:
            y_line += average_coefficients["beta"] * period_data["beta"].mean()

        ax.scatter(
            period_data["downside_beta"], period_data["excess_return"],
            alpha=0.12, s=8, edgecolors="none",
        )
        ax.plot(x_line, y_line, color="crimson", linewidth=2)
        ax.axhline(0, color="black", linewidth=0.8, alpha=0.7)
        ax.set(
            title=period_name,
            xlabel="Downside beta estimated through prior month",
            ylabel="Next-month stock excess return",
        )
        ax.grid(alpha=0.2)
    fig.suptitle(model_name)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

print("Regression outcome window (inclusive):")
for period_name, (period_start, period_end) in TIME_PERIODS.items():
    print(f"- {period_name}: {period_start:%Y-%m} through {period_end:%Y-%m}")
print("All regression outcome months are July 2009-January 2020; beta predictors are lagged one month.")
print("Historical data before 2006 may be used only in the trailing beta lookback.")
print(
    f"Inference: monthly Fama-MacBeth coefficients with Newey-West/HAC standard errors "
    f"({HAC_LAGS} lags); p-values use the normal approximation."
)
display(model_summary, regression_summary)

for period_name in TIME_PERIODS:
    for model_name in model_specs:
        result = regression_summary.loc[
            regression_summary["period"].eq(period_name)
            & regression_summary["model"].eq(model_name)
            & regression_summary["term"].eq("downside_beta")
        ].iloc[0]
        significance = (
            "statistically significant at 5%"
            if result["p_value_normal_approx"] < 0.05
            else "not statistically significant at 5%"
        )
        print(
            f"{period_name} | {model_name}: downside-beta coefficient="
            f"{result['mean_monthly_coefficient']:.6f}, "
            f"95% CI=({result['ci_95_low']:.6f}, {result['ci_95_high']:.6f}), "
            f"p={result['p_value_normal_approx']:.4f}; {significance}."
        )